# INFO 230 Mini-Project 2: Marvel Universe Social Network 

This notebook walks through a full network analysis cultural analytics workflow applied to the <span style="color: red;">[INSERT LINK TO DATASET]</span> from Kaggle. 
- explain the dataset structure and the central research question 

## <span style="color: blue;">1. Motivation</span>

The Marvel Universe is one of the most expansive and deliberately constructed  narrative universes in the history of cultural production. Marvel has been publishing comics since the 1960s, and over those six decades, thousands of heroes have shared panels, storylines, and team-ups. Notably, every one of those co-appearances was an editorial choice. For example, someone decided that Spider-Man and Captain America should be in the same issue, that Wolverine should cross over into the Avengers, that certain characters would sit at the center of the universe, while others stayed on the margins. 

Therefore, this project asks whether we can see those choices in the data. Specifically, this project will explore the following primary questions. 
1. **Does network centrality reflect cultural prominence?**
    - Do the heroes who show up most centrally in the network match the ones we'd actually recognize as the "pillars" of Marvel? 
2. **Do communities in the network correspond to real Marvel teams?** 
    - Can the algorithm recover the X-Men, the Avengers, the Fantastic Four on its own, or is the network messier than the official team structure suggests? In other words, does the data tell a more complicated story than the teams Marvel actually intended?
3. **Does frequency of co-appearance tell a different story than just co-appearance count?** 
    - Think of it like a social network... some people know everyone but only barely. Others have a tight-knit group they're always around. So, weighting edges by how ofter two heroes appear together (not just whether they do) will let us tell those two patterns apart.  

### General Setup
- *Imports and configuration. Community detection uses Louvain when available, falling back to greedy modularity otherwise: this pattern was adapted from the course Holmes co-occurrence notebook.*


In [1]:
import pandas as pd
import numpy as np
import networkx as nx
from networkx.algorithms import bipartite
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from collections import defaultdict, Counter
import warnings
import os

warnings.filterwarnings("ignore")

print(f"NetworkX : {nx.__version__}")
print(f"Pandas : {pd.__version__}")
print(f"NumPy : {np.__version__}")

NetworkX : 3.6.1
Pandas : 3.0.0
NumPy : 2.3.5


In [2]:
try:
    from networkx.algorithms.community import louvain_communities
    HAS_LOUVAIN = True
except ImportError:
    from networkx.algorithms.community import greedy_modularity_communities
    HAS_LOUVAIN = False

method_name = "Louvain" if HAS_LOUVAIN else "Greedy Modularity"
print(f"Community detection method: {method_name}")

Community detection method: Louvain


In [3]:
# Configuration 
DATA_DIR = "../data"
NET_OUT = "./network_output"
os.makedirs(NET_OUT, exist_ok=True)

MIN_EDGE_WEIGHT = 2
TOP_N_NODES = 150
SEED = 230

print("Configuration ready:")
print(f"\tData directory : {DATA_DIR}")
print(f"\tOutput directory : {NET_OUT}")
print(f"\tMin edge weight : {MIN_EDGE_WEIGHT}")
print(f"\tMax display nodes: {TOP_N_NODES}")
print(f"\tSeed : {SEED}")

Configuration ready:
	Data directory : ../data
	Output directory : ./network_output
	Min edge weight : 2
	Max display nodes: 150
	Seed : 230


### Load in the data

In [4]:
# load all three raw files 
nodes = pd.read_csv("../data/nodes.csv")
edges = pd.read_csv("../data/edges.csv")
hero_net = pd.read_csv("../data/hero-network.csv")

# quick inspection 
print("nodes.csv :", nodes.shape)
print("edges.csv :", edges.shape)
print("hero-network :", hero_net.shape)

nodes.csv : (19090, 2)
edges.csv : (96104, 2)
hero-network : (574467, 2)


## <span style="color: blue;">2. Dataset Development</span>
The dataset for this project is the <span style="color: red;">[Marvel Universe Social Network]</span> from Kaggle, originally compiled from Marvel's published comic index. It came with three files: 
- ```nodes.csv``` lists every node and whether it's a hero or a comic issue. 
- ```edges.csv``` is the two-sided network (i.e., each row is one hero appearing in one comic issue). 
- ```hero-network.csv``` is a pre-projected hero-to-hero network where two heroes are connected if they appeared in the same issue.

***Note that this dataset is ready-made, but it needs meaningful work (as follows) before it is usable for analysis:***


### Development Step 1: Build a weighted edge list from scratch

In [5]:
# count how many times each hero pair co-appears 
    # this becomes the edge weight 
# raw hero-network.csv lists each shared comic as a separate row 
    # so grouping and counting gives a proper weighted edge list 

hero_weighted = (
    hero_net.groupby(["hero1", "hero2"]).size().reset_index(name="weight")
)

print("Unique hero pairs (edges):", len(hero_weighted))
print("Weight distribution:")
print(hero_weighted["weight"].describe().round(2))

Unique hero pairs (edges): 224181
Weight distribution:
count    224181.00
mean          2.56
std           7.90
min           1.00
25%           1.00
50%           1.00
75%           2.00
max        1275.00
Name: weight, dtype: float64


### Development Step 2: Apply a minimum edge weight threshold 
- NOTE: A weight of 1 means two heroes shared exactly one comic issue (likely coincidental background appearance rather than a meaningful relationship). 
    - Therefore I will filter to a minimum edge weight of 2 to keep only these pairs (i.e., those with at least two co-appearances).

In [6]:
# filter the data 
hero_filtered = hero_weighted[hero_weighted["weight"] >= MIN_EDGE_WEIGHT].copy()
print(f"Edges before filtering : {len(hero_weighted):,}")
print(f"Edges after filtering : {len(hero_filtered):,}")
print(f"Edges removed : {len(hero_weighted) - len(hero_filtered):,}")


Edges before filtering : 224,181
Edges after filtering : 85,352
Edges removed : 138,829


### Development Step 3: Flag truncated hero names 
- Some hero names in the dataset are cut off at 20 characters (e.g., ABOMINATION/EMIL BLO). 
    - This step will flag these so they're documented as a known data quality limitation rather than silently passed over. 

In [7]:
# names truncated at exactly 20 characters are a known encoding issue in this dataset
truncated = nodes[nodes["type"] == "hero"]["node"].str.len() == 20
print(f"Potentially truncated hero names: {truncated.sum()}")

Potentially truncated hero names: 1351


### Development Step 4: Validation of all datasets that they are good to proceed with

In [8]:
# quick validation pass across all three files before building any graphs
    # this confirms no null values slipped through and documents basic shape in a clean summary output 

print("=== nodes.csv ===")
print(f"\tShape: {nodes.shape}")
print(f"\tTypes: {nodes['type'].value_counts().to_dict()}")
print(f"\tNulls: {nodes.isnull().sum().to_dict()}")

print("\n=== edges.csv ===")
print(f"\tShape: {edges.shape}")
print(f"\tUnique heroes: {edges['hero'].nunique()}")
print(f"\tUnique comics: {edges['comic'].nunique()}")
print(f"\tNulls: {edges.isnull().sum().to_dict()}")

print("\n=== hero-network.csv (after weighting + filtering) ===")
print(f"\tRaw rows: {len(hero_net):,}")
print(f"\tUnique pairs: {len(hero_weighted):,}")
print(f"\tAfter filter: {len(hero_filtered):,}")
print(f"\tNulls: {hero_filtered.isnull().sum().to_dict()}")


=== nodes.csv ===
	Shape: (19090, 2)
	Types: {'comic': 12651, 'hero': 6439}
	Nulls: {'node': 0, 'type': 0}

=== edges.csv ===
	Shape: (96104, 2)
	Unique heroes: 6439
	Unique comics: 12651
	Nulls: {'hero': 0, 'comic': 0}

=== hero-network.csv (after weighting + filtering) ===
	Raw rows: 574,467
	Unique pairs: 224,181
	After filter: 85,352
	Nulls: {'hero1': 0, 'hero2': 0, 'weight': 0}


## <span style="color: blue;">3. Analytical Approach & Metric Justification</span>

The goal in this step of the project is not just to describe the network, but also ask whether the structure of the Marvel co-appearance network reflects meaningful patterns in how Marvel has constructed its universe. The metrics discussed below were chosen because each one answers a distinct cultural question and not just a technical one. 

**NOTE Two Graphs, Not Just One:** This analysis is run on both weighted and unweighted versions of the hero-hero network. Keep in mind that the unweighted graph treats any co-appearance relationship as equal regardless of the frequency of such. In contrast, the weighted graph scales edge strength by how many times two heroes have shared a comic issue. Therefore, comparing centrality rankings across both these versions will tell us whether frequency of collaboration changes who looks "important", or whether the top heroes dominate either way.

**Metric 1 - Degree Centrality:** How many unique heroes does each hero co-appear with? This is the most direct measure of connectivity and serves us as a sanity check. For instance, if Captain America and Spider-Man don't rank near the top, then something is wrong with the data. Overall culturally, high degree = broad reach across the universe. 

**Metric 2 - Between-ness Centrality:** Which heroes sit on the shortest paths between other heroes who wouldn't otherwise be connected? This identifies bridge character, for instance heroes like Wolverine who cross team lines and link otherwise separate corners of the Marvel Universe. This is arguably the most culturally interesting metric because it captures narrative role and not just general popularity. 

**Metric 3 - Eigenvector Centrality:** This measures not just how many connections a hero has but also how well-connected these connections are. For instance, a hero who only appears with other major heroes will score higher than one who appears with obscure characters the same number of times, thereby capturing prestige rather than pure reach. 

**Community Detection- Louvain algorithm:** Groups heroes into communities based on density of shared connections. The key question here is whether these algorithmically detected communities map onto real Marvel teams. Therefore, with this metric I will compare the top communities against known groupings like X-Men, Avengers, and Fantastic Four. 

**Comic-Level Analysis:** Before looking at how heroes connect to each other, it's worth stepping back and asking which comic issues brought the most heroes together in the first place? Therefore, this part of the analysis will look at the comics themselves as the unit of interest, i.e., something that gets lost once you collapse everything down to just hero-to-hero connections. 


#### Part 1: Build the weighted hero-hero graph

In [9]:
#  each edge carries a "weight" attribute = number of shared comic appearances
G_weighted = nx.Graph() 

for _, row in hero_filtered.iterrows():
    G_weighted.add_edge(row["hero1"], row["hero2"], weight=row["weight"])

print("Weighted graph")
print(f"\tNodes: {G_weighted.number_of_nodes():,}")
print(f"\tEdges: {G_weighted.number_of_edges():,}")



Weighted graph
	Nodes: 4,465
	Edges: 62,128


#### Part 2: Build the unweighted version for comparison

In [10]:
# same edges, but weight is ignored (i.e., every connection treated as equal)

G_unweighted = nx.Graph(G_weighted)
for u, v in G_unweighted.edges(): 
    G_unweighted[u][v]["weight"] = 1 

print("Unweighted graph")
print(f"\tNodes: {G_unweighted.number_of_nodes}")
print(f"\tEdges: {G_unweighted.number_of_edges}")

Unweighted graph
	Nodes: <bound method Graph.number_of_nodes of <networkx.classes.graph.Graph object at 0x136b4fb10>>
	Edges: <bound method Graph.number_of_edges of <networkx.classes.graph.Graph object at 0x136b4fb10>>


#### Part 3: Compute centrality metrics on both graphs 

In [11]:
# note that between-ness is an expensive computation on large graphs 
    # use a sample of 500 nodes to approximate (standard practice for networks this size)

print("Computing degree centrality...")
deg_weighted = nx.degree_centrality(G_weighted)
deg_unweighted = nx.degree_centrality(G_unweighted)

print("Computing betweenness centrality (approximated, k=500)...")
btw_weighted = nx.betweenness_centrality(G_weighted, weight="weight", k=500, seed=SEED)
btw_unweighted = nx.betweenness_centrality(G_unweighted, k=500, seed=SEED)

print("Computing eigenvector centrality...")
eig_weighted = nx.eigenvector_centrality(G_weighted, weight="weight", max_iter=1000)
eig_unweighted = nx.eigenvector_centrality(G_unweighted, max_iter=1000)

print("Done.")

Computing degree centrality...
Computing betweenness centrality (approximated, k=500)...
Computing eigenvector centrality...
Done.


#### Part 4: Compile into a metrics dataframe
- *adapted from the centrality metrics table pattern in the course Holmes co-occurrence notebook*

In [12]:
heroes_list = list(G_weighted.nodes())

metrics_df = pd.DataFrame({
    "hero" : heroes_list,
    "degree_weighted" : [deg_weighted[h] for h in heroes_list],
    "degree_unweighted" : [deg_unweighted[h] for h in heroes_list],
    "betweenness_wtd" : [btw_weighted[h] for h in heroes_list],
    "betweenness_unwtd" : [btw_unweighted[h] for h in heroes_list],
    "eigenvector_wtd" : [eig_weighted[h] for h in heroes_list],
    "eigenvector_unwtd" : [eig_unweighted[h] for h in heroes_list],
    "raw_degree" : [G_weighted.degree(h) for h in heroes_list],
}).sort_values("degree_weighted", ascending=False).reset_index(drop=True)

print("Top 10 by weighted degree:")
print(metrics_df[["hero","degree_weighted","betweenness_wtd","eigenvector_wtd"]].head(10))

Top 10 by weighted degree:
                   hero  degree_weighted  betweenness_wtd  eigenvector_wtd
0       CAPTAIN AMERICA         0.225806         0.060866         0.010815
1  SPIDER-MAN/PETER PAR         0.195789         0.051988         0.002985
2  IRON MAN/TONY STARK          0.170251         0.027778         0.006454
3      WOLVERINE/LOGAN          0.159722         0.033317         0.002875
4  THING/BENJAMIN J. GR         0.149642         0.024430         0.005661
5  THOR/DR. DONALD BLAK         0.148297         0.023400         0.005265
6  SCARLET WITCH/WANDA          0.145833         0.019366         0.006129
7  MR. FANTASTIC/REED R         0.142697         0.030277         0.005415
8  HUMAN TORCH/JOHNNY S         0.138441         0.019567         0.005508
9                  HAWK         0.133065         0.034386         0.004645


#### Part 5: Community detection

In [13]:
if HAS_LOUVAIN:
    communities = louvain_communities(G_weighted, weight="weight", seed=SEED)
    method_used = "Louvain"
else:
    communities = greedy_modularity_communities(G_weighted, weight="weight")
    method_used = "Greedy Modularity"

community_map = {}
for i, comm in enumerate(communities):
    for hero in comm:
        community_map[hero] = i

metrics_df["community"] = metrics_df["hero"].map(community_map)

print(f"Method: {method_used}")
print(f"Communities detected: {len(communities)}")
print("\nTop 5 communities by size:")
comm_sizes = pd.Series([len(c) for c in communities]).sort_values(ascending=False)
print(comm_sizes.head())

Method: Louvain
Communities detected: 36

Top 5 communities by size:
1     901
8     697
13    659
0     418
2     408
dtype: int64


#### Build the hero-comic network from edges.csv 


In [14]:
# heroes and comics are two distinct node types 
    # tag each node so NetworkX knows which side it belongs to 
        # bipartite=0/1 tags are required by NetworkX to recognize the two sides
            # 0 = heroes, 1 = comics

B = nx.Graph()

heroes = edges["hero"].unique()
comics = edges["comic"].unique()

B.add_nodes_from(heroes, bipartite=0)    # heroes 
B.add_nodes_from(comics, bipartite=1)    # comics 

for _, row in edges.iterrows(): 
    B.add_edge(row["hero"], row["comic"])

print("Hero-comic network:")
print(f"\tHero nodes: {len(heroes):,}")
print(f"\tComic nodes: {len(comics):,}")
print(f"\tEdges: {B.number_of_edges():,}")
print(f"Valid two-sided structure: {nx.is_bipartite(B)}")


Hero-comic network:
	Hero nodes: 6,439
	Comic nodes: 12,651
	Edges: 96,104
Valid two-sided structure: True


## <span style="color: blue;">4. Results & Visualizations</span>


## <span style="color: blue;">5. Discussion</span>


## <span style="color: blue;">6. Limitations </span>


## <span style="color: blue;">7. Further Study</span>
